# PubMed RAG: Dense Embedding & Chroma Vector Store Pipeline

## Settings

In [1]:
import os
import sys
import time
from pathlib import Path

import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# -------------------------
# Load environment variables
# -------------------------
load_dotenv(find_dotenv())

# -------------------------
# Add project root to path
# -------------------------
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.chroma_manager import ChromaManager

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = Path(os.environ.get("DATA_DIR", "/workspace/data"))

CHUNK_DIR = DATA_DIR / "chunked_docs"
VECTOR_DIR = DATA_DIR / "vector_store"
BM25_DIR = DATA_DIR / "bm25_index"

# -------------------------
# Ensure directories exist
# -------------------------
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
BM25_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Ensure directories exist
# -------------------------
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Chunk dir: {CHUNK_DIR}")
print(f"Vector DB dir: {VECTOR_DIR}")

2026-03-12 12:07:11.309990019 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


SEED: 42
CUDA available: True
PyTorch version: 2.2.0
Chunk dir: /workspace/data/chunked_docs
Vector DB dir: /workspace/data/vector_store


## Helpers

In [9]:
import spacy
from tqdm import tqdm

# -------------------------
# Load spaCy model
# -------------------------

nlp = spacy.load(
    "en_core_sci_sm",
    disable=["parser", "ner", "tagger", "lemmatizer"]
)

# -------------------------
# Helper: tokenize a spaCy doc
# -------------------------

def tokenize_spacy_doc(doc):
    return [
        token.text.lower()
        for token in doc
        if not token.is_punct and not token.is_space
    ]


# -------------------------
# Helper: tokenize list of texts
# -------------------------

def spacy_tokenize_texts(texts, batch_size=64):
    tokenized = []

    for doc in tqdm(
        nlp.pipe(texts, batch_size=batch_size),
        total=len(texts),
        desc="spaCy tokenizing"
    ):
        tokenized.append(tokenize_spacy_doc(doc))

    return tokenized

## 1. Section-Aware Chunking (400 Tokens)

In [23]:
section_chunks_df = pd.read_csv(f"{CHUNK_DIR}/chunks_section_400token.csv")

In [40]:
db = ChromaManager(
    base_directory=VECTOR_DIR,
    chunk_strategy="section_400token",
    embedding_model="pubmedbert",
    collection_name="medical_rag"
)


Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/section_400token/pubmedbert


In [25]:
db.reset()

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection reset.


### Embedding Generation and Vector Database Construction

In [26]:
start = time.time()

# db_section.add_chunks(section_chunks_df)

elapsed = time.time() - start

print(f"\nInserted {len(section_chunks_df)} chunks")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Avg time per chunk: {elapsed/len(section_chunks_df):.4f} sec")

Preparing chunks:   0% 0/109234 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Preparing chunks:   0% 256/109234 [00:03<22:29, 80.77it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Preparing chunks:   0% 512/109234 [00:05<20:54, 86.69it/s]

...

Preparing chunks: 100% 108800/109234 [23:36<00:05, 76.02it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Preparing chunks: 100% 109234/109234 [23:42<00:00, 76.80it/s]


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Finished inserting all chunks.

Inserted 109234 chunks
Total time: 1426.36 seconds
Avg time per chunk: 0.0131 sec


In [42]:
print("Collection count:", db_section.collection.count())

Collection count: 109234


### Inspect a few stored documents

In [29]:
sample = db_section.collection.get(
    limit=5,
    include=["documents", "metadatas", "embeddings"]
)

for i in range(len(sample["ids"])):
    print("\nID:", sample["ids"][i])
    print("PMID:", sample["metadatas"][i]["pmid"])
    print("Section:", sample["metadatas"][i]["section"])
    print("Text preview:", sample["documents"][i][:200])
    print("Embedding length:", len(sample["embeddings"][i]))

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



ID: 10024311_ABSTRACT_0
PMID: 10024311
Section: ABSTRACT
Text preview: Vascular endothelial cells regulate vascular smooth muscle tone through Ca2+-dependent production and release of vasoactive molecules. Phospholamban (PLB) is a 24- to 27-kDa phosphoprotein that modula
Embedding length: 768

ID: 10024311_ABSTRACT_1
PMID: 10024311
Section: ABSTRACT
Text preview: . These data indicate that PLB is present and modulates vascular function as a result of its actions in endothelial cells. The presence of PLB in endothelial cells opens new fields for investigation o
Embedding length: 768

ID: 10024311_TITLE
PMID: 10024311
Section: TITLE
Text preview: Phospholamban is present in endothelial cells and modulates endothelium-dependent relaxation. Evidence from phospholamban gene-ablated mice.
Embedding length: 768

ID: 10024707_AIM_0
PMID: 10024707
Section: AIM
Text preview: To evaluate a simple, ear acupuncture treatment for the cessation of smoking.
Embedding length: 768

ID: 10024707_BACKGRO

### Run a retrieval test

In [41]:
query = "What causes Alzheimer's disease?"

results = db_section.query(query, n_results=3)

for i in range(len(results["documents"][0])):
    print("\nResult", i+1)
    print(results["documents"][0][i][:300])
    print("PMID:", results["metadatas"][0][i]["pmid"])


Result 1
The Infectious Etiology of Alzheimer's Disease.
PMID: 28294067

Result 2
Alzheimer's disease is thought to be caused by increased formations of neurotoxic amyloid beta (A beta) peptides, which give rise to the hallmark amyloid plaques. Therefore, pharmacological agents that reduce A beta formation may be of therapeutic benefit.
PMID: 19527190

Result 3
Mutations in the amyloid precursor protein gene were the first to be recognized as a cause of Alzheimer's disease (AD).
PMID: 20523046


### Check similarity scores

In [31]:
for i, dist in enumerate(results["distances"][0]):
    print(f"Result {i+1} distance:", dist)

Result 1 distance: 20.033445358276367
Result 2 distance: 21.020559310913086
Result 3 distance: 23.378067016601562


### Build BM25

In [32]:
import time
from rank_bm25 import BM25Okapi

# -------------------------
# Prepare documents
# -------------------------

docs = section_chunks_df["chunk_text"].tolist()

start_total = time.time()

# -------------------------
# Tokenize with spaCy
# -------------------------

start_token = time.time()

tokenized_docs = spacy_tokenize_texts(docs)

end_token = time.time()

print(f"Tokenization completed: {len(tokenized_docs)} docs")
print(f"Tokenization time: {end_token - start_token:.2f} seconds")

# -------------------------
# Build BM25 index
# -------------------------

start_bm25 = time.time()

bm25_section = BM25Okapi(tokenized_docs)

end_bm25 = time.time()

print("BM25 index built:", len(tokenized_docs))
print(f"BM25 build time: {end_bm25 - start_bm25:.2f} seconds")

# -------------------------
# Total time
# -------------------------

end_total = time.time()

print(f"Total pipeline time: {end_total - start_total:.2f} seconds")

spaCy tokenizing: 100% 109234/109234 [04:30<00:00, 404.16it/s]


Tokenization completed: 109234 docs
Tokenization time: 270.41 seconds
BM25 index built: 109234
BM25 build time: 2.77 seconds
Total pipeline time: 273.18 seconds


### Save BM25 Index

In [35]:
import pickle

bm25_path = BM25_DIR / "section_400token.pkl"

with open(bm25_path, "wb") as f:
    pickle.dump(bm25_section, f)

print("BM25 index saved to:", bm25_path)

BM25 index saved to: /workspace/data/bm25_index/section_400token.pkl


### Verify BM25 Retrieval

In [36]:
query = "amyloid beta plaque formation in Alzheimer's disease"

tokenized_query = spacy_tokenize_texts([query])[0]

scores = bm25_section.get_scores(tokenized_query)

import numpy as np

top_k = 10
top_idx = np.argsort(scores)[-top_k:][::-1]

results = section_chunks_df.iloc[top_idx]

results[["pmid", "section", "chunk_text"]]

spaCy tokenizing: 100% 1/1 [00:00<00:00, 363.90it/s]


,pmid,section,chunk_text
101172,29080524,TITLE,Oxidative stress and the amyloid beta peptide ...
25981,19527190,BACKGROUND,Alzheimer's disease is thought to be caused by...
22248,18191617,TITLE,Imaging of amyloid beta in Alzheimer's disease...
11631,20375655,TITLE,Safety and changes in plasma and cerebrospinal...
3983,21459773,TITLE,Impaired mitochondrial dynamics and abnormal i...
23055,21342558,BACKGROUND,Beta-site amyloid precursor protein cleaving e...
14659,20816785,ABSTRACT,There is strong evidence that intracellular ca...
80257,34198582,ABSTRACT,A large body of clinical and nonclinical evide...
97601,38095822,ABSTRACT,The approval of lecanemab by the US Food and D...
101158,35892582,TITLE,Human Palatine Tonsils Are Linked to Alzheimer...


## 2. Fixed-Length Chunking (500 Characters)

In [46]:
fixed_chunks_df = pd.read_csv(f"{CHUNK_DIR}/chunks_fixed500char.csv")

In [47]:
db_fixed = ChromaManager(
    base_directory=VECTOR_DIR,
    chunk_strategy="fixed_500char",
    embedding_model="pubmedbert",
    collection_name="medical_rag"
)


Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/fixed_500char/pubmedbert


### Embedding Generation and Vector Database Construction

In [48]:
db_fixed.reset()

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection reset.


In [49]:
import time

start = time.time()

db_fixed.add_chunks(fixed_chunks_df)

elapsed = time.time() - start

print(f"\nInserted {len(fixed_chunks_df)} chunks")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Avg time per chunk: {elapsed/len(fixed_chunks_df):.4f} sec")

Preparing chunks:   0% 0/176626 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Preparing chunks:   0% 256/176626 [00:03<42:22, 69.37it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Preparing chunks:   0% 512/176626 [00:07<41:55, 70.02it/s]

...

Preparing chunks: 100% 176128/176626 [32:20<00:05, 85.22it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Preparing chunks: 100% 176626/176626 [32:23<00:00, 90.90it/s]


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Finished inserting all chunks.

Inserted 176626 chunks
Total time: 1944.80 seconds
Avg time per chunk: 0.0110 sec


In [56]:
print("Collection count:", db_fixed.collection.count())

Collection count: 176626


### Inspect a few stored documents

In [50]:
sample = db_fixed.collection.get(
    limit=5,
    include=["documents", "metadatas", "embeddings"]
)

for i in range(len(sample["ids"])):
    print("\nID:", sample["ids"][i])
    print("PMID:", sample["metadatas"][i]["pmid"])
    print("Section:", sample["metadatas"][i]["section"])
    print("Text preview:", sample["documents"][i][:200])
    print("Embedding length:", len(sample["embeddings"][i]))

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



ID: 10024311_F500_0
PMID: 10024311
Section: ABSTRACT
Text preview: Vascular endothelial cells regulate vascular smooth muscle tone through Ca2+-dependent production and release of vasoactive molecules. Phospholamban (PLB) is a 24- to 27-kDa phosphoprotein that modula
Embedding length: 768

ID: 10024311_F500_1
PMID: 10024311
Section: ABSTRACT
Text preview: actility in these muscles. In the present study, we report the existence of PLB in the vascular endothelium, a nonmuscle tissue, and provide functional data on PLB regulation of vascular contractility
Embedding length: 768

ID: 10024311_F500_2
PMID: 10024311
Section: ABSTRACT
Text preview: of nitric oxide on the smooth muscle, because sodium nitroprusside-mediated relaxation in either denuded or endothelium-intact aortas was unaffected by PLB ablation. Relative to denuded vessels, relax
Embedding length: 768

ID: 10024311_F500_3
PMID: 10024311
Section: ABSTRACT
Text preview: a endothelial cells were isolated. Both reverse transcripta

### Run a retrieval test

In [51]:
query = "What causes Alzheimer's disease?"

results = db_fixed.query(query, n_results=3)

for i in range(len(results["documents"][0])):
    print("\nResult", i+1)
    print(results["documents"][0][i][:300])
    print("PMID:", results["metadatas"][0][i]["pmid"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Result 1
Alzheimer's disease (AD) is the most frequent cause of dementia in the elderly. Few cases are familial (FAD), due to autosomal dominant mutations in presenilin-1 (PS1), presenilin-2 (PS2) or amyloid precursor protein (APP). The three proteins are involved in the generation of amyloid-beta (Aβ) pepti
PMID: 31606858

Result 2
Alzheimer's disease (AD) is the primary cause of dementia in the elderly. It remains incurable and poses a huge socio-economic challenge for developed countries with an aging population. AD manifests by progressive decline in cognitive functions and alterations in behaviour, which are the result of 
PMID: 27023706

Result 3
Alzheimer's disease (AD) is the most common cause of dementia in humans. A pathological hallmark in the brain of an AD patient is extracellular amyloid plaques formed by accumulated beta-amyloid protein (Abeta), a metabolic product of amyloid precursor protein (APP). Studies have revealed a strong g
PMID: 16426772


### Check similarity scores

In [52]:
for i, dist in enumerate(results["distances"][0]):
    print(f"Result {i+1} distance:", dist)

Result 1 distance: 20.541053771972656
Result 2 distance: 20.69760513305664
Result 3 distance: 21.04430389404297


### Build BM25

In [57]:
import time
from rank_bm25 import BM25Okapi

# -------------------------
# Prepare documents
# -------------------------

docs = fixed_chunks_df["chunk_text"].tolist()

start_total = time.time()

# -------------------------
# Tokenize with spaCy
# -------------------------

start_token = time.time()

tokenized_docs = spacy_tokenize_texts(docs)

end_token = time.time()

print(f"Tokenization completed: {len(tokenized_docs)} docs")
print(f"Tokenization time: {end_token - start_token:.2f} seconds")

# -------------------------
# Build BM25 index
# -------------------------

start_bm25 = time.time()

bm25_fixed = BM25Okapi(tokenized_docs)

end_bm25 = time.time()

print("BM25 index built:", len(tokenized_docs))
print(f"BM25 build time: {end_bm25 - start_bm25:.2f} seconds")

# -------------------------
# Total time
# -------------------------

end_total = time.time()

print(f"Total pipeline time: {end_total - start_total:.2f} seconds")

spaCy tokenizing: 100% 176626/176626 [05:03<00:00, 582.68it/s]


Tokenization completed: 176626 docs
Tokenization time: 303.30 seconds
BM25 index built: 176626
BM25 build time: 2.91 seconds
Total pipeline time: 306.21 seconds


### Save BM25 Index

In [58]:
import pickle

# -------------------------
# Save BM25 index (fixed 500 chunk)
# -------------------------

bm25_path = BM25_DIR / "fixed_500char.pkl"

with open(bm25_path, "wb") as f:
    pickle.dump(bm25_fixed, f)

print("BM25 index saved to:", bm25_path)

BM25 index saved to: /workspace/data/bm25_index/fixed_500char.pkl


### Verify BM25 Retrieval

In [66]:
query = "amyloid beta plaque formation in Alzheimer's disease"

tokenized_query = spacy_tokenize_texts([query])[0]

scores = bm25_fixed.get_scores(tokenized_query)

import numpy as np

top_k = 10
top_idx = np.argsort(scores)[-top_k:][::-1]

results = fixed_chunks_df.iloc[top_idx]

results[["pmid", "section", "chunk_text"]]

spaCy tokenizing: 100% 1/1 [00:00<00:00, 380.02it/s]


,pmid,section,chunk_text
164358,29080524,TITLE,Oxidative stress and the amyloid beta peptide ...
37154,18191617,TITLE,Imaging of amyloid beta in Alzheimer's disease...
19386,20375655,TITLE,Safety and changes in plasma and cerebrospinal...
6653,21459773,TITLE,Impaired mitochondrial dynamics and abnormal i...
38629,21342558,ABSTRACT,ry species thought to be responsible for the n...
164338,35892582,TITLE,Human Palatine Tonsils Are Linked to Alzheimer...
37115,24199960,ABSTRACT,Cholesterol is implicated in the development o...
164425,26657517,ABSTRACT,"Despite amyloid plaques, consisting of insolub..."
164388,32183293,TITLE,Potential Role of Venular Amyloid in Alzheimer...
24690,20816785,ABSTRACT,There is strong evidence that intracellular ca...


In [64]:
import pickle

bm25_section = pickle.load(open(BM25_DIR / "section_400token.pkl", "rb"))
bm25_fixed500 = pickle.load(open(BM25_DIR / "fixed_500char.pkl", "rb"))

In [65]:
print("Section chunks:", len(bm25_section.doc_len))
print("Fixed500 chunks:", len(bm25_fixed500.doc_len))

Section chunks: 109234
Fixed500 chunks: 176626
